# Isochrone CMD Plots
Generates color magnitude diagrams with MIST isochrone overlays for the four
clusters with multiple seismic detections: NGC 752, Theia 6046, Casado Alessi 1,
and Theia 844. Plots show seismic age range (solid) and literature age range (dashed).

> **Revision update (2026):** rebuilt for the revised manuscript. Uses the stock `isochrones` API (the old `interp_value` monkeypatch that scrambled the EEP order was removed), applies a parallax + proper-motion sigma-clip to show clean members, and overlays two seismic-age curves (green) plus three literature-age curves (dashed orange) per cluster. Outputs cleared; re-run to regenerate.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from isochrones import get_ichrone
import warnings; warnings.filterwarnings('ignore')

STAR_FILE = '../../GitHub/Isochrones/final!_errors.csv'  # adjust path as needed
df = pd.read_csv(STAR_FILE, low_memory=False)
cluster_col = 'cluster_name'
outdir = '../../Downloads/revision_1'

# Gaia DR3 extinction coefficients
K_G, K_BP, K_RP = 2.74, 3.37, 2.04

# Stock isochrones API (no monkeypatch: the old interp_value patch scrambled the
# EEP ordering and produced the spurious "curl").
mist = get_ichrone('mist', bands=['G', 'BP', 'RP'])

def robust_stats(series):
    s = pd.to_numeric(series, errors='coerce')
    med = np.nanmedian(s); mad = np.nanmedian(np.abs(s - med))
    return med, (1.4826 * mad if mad > 0 else np.nan)

def get_cluster_data(df, pattern):
    """color, gmag, good for kinematically-clean members (parallax + PM sigma-clip, max|z|<2)."""
    dfc = df[df[cluster_col].astype(str).str.contains(pattern, case=False, na=False)].copy()
    color = (dfc['BPmag'] - dfc['RPmag']).to_numpy()
    gmag = pd.to_numeric(dfc['Gmag'], errors='coerce').to_numpy()
    good = np.isfinite(color) & np.isfinite(gmag)
    zs = []
    for c in ['Plx', 'pmRA', 'pmDE']:
        med, sig = robust_stats(dfc[c])
        x = pd.to_numeric(dfc[c], errors='coerce')
        zs.append(np.abs((x - med) / sig).to_numpy() if (np.isfinite(sig) and sig > 0)
                  else np.full(len(dfc), np.nan))
    max_abs_z = np.nanmax(np.vstack(zs), axis=0)
    member = np.isfinite(max_abs_z) & (max_abs_z < 2.0)
    good = good & member
    plx = pd.to_numeric(dfc['Plx'], errors='coerce').to_numpy()[good]
    dm = 5 * np.log10(1000.0 / np.nanmedian(plx)) - 5 if np.nanmedian(plx) > 0 else np.nan
    print(f'  {pattern}: {good.sum()} clean members | dm={dm:.2f}')
    return color, gmag, good


In [ ]:
def plot_isochrones(color, gmag, good, label, seismic_ages, lit_ages, feh, ebv, dm, xlim, ylim, savefile):
    """Solid green = seismic age range (2 curves); dashed orange = literature range (3 curves)."""
    fig, ax = plt.subplots(figsize=(9, 8))
    ax.scatter(color[good], gmag[good], s=15, color='gray', edgecolors='k',
               linewidth=0.5, alpha=0.75, label=label, zorder=1)
    def iso_curve(age):
        iso = mist.isochrone(np.log10(age * 1e9), feh)
        mabs = iso['G_mag'].to_numpy(); keep = (mabs > -3) & (mabs < 10)
        col = (iso['BP_mag'] - iso['RP_mag']).to_numpy()[keep] + (K_BP - K_RP) * ebv
        return col, mabs[keep] + dm + K_G * ebv
    for k, age in enumerate(seismic_ages):
        c, m = iso_curve(age)
        lab = fr'Seismic (this work): {min(seismic_ages):.2f}--{max(seismic_ages):.2f} Gyr' if k == 0 else None
        ax.plot(c, m, color='#2ca02c', lw=2.6, ls='-', zorder=4, label=lab)
    for k, age in enumerate(lit_ages):
        c, m = iso_curve(age)
        lab = fr'Literature: {min(lit_ages):.2f}--{max(lit_ages):.2f} Gyr' if k == 0 else None
        ax.plot(c, m, color='#ff7f0e', lw=2.0, ls='--', alpha=0.85, zorder=3, label=lab)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_xlabel(r'$G_{\rm BP} - G_{\rm RP}$ (mag)'); ax.set_ylabel(r'$G$ (mag)')
    ax.legend(fontsize=10, loc='upper left'); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(f'{outdir}/{savefile}', dpi=200, bbox_inches='tight'); plt.show()

CLUSTERS = [
    dict(pattern='NGC_752',         label='NGC 752',         seismic_ages=[1.45, 1.71], lit_ages=[1.00, 1.30, 1.60], feh=0.0,   ebv=0.04, dm=8.22, xlim=(0.0, 1.5), ylim=(14, 8),   savefile='ngciso.png'),
    dict(pattern='Theia_6046',      label='Theia 6046',      seismic_ages=[2.59, 6.69], lit_ages=[2.50, 3.50, 4.50], feh=-0.13, ebv=0.30, dm=9.61, xlim=(0.2, 2.5), ylim=(17, 9),   savefile='theiaiso.png'),
    dict(pattern='Casado-Alessi_1', label='Casado-Alessi 1', seismic_ages=[0.88, 1.09], lit_ages=[0.69, 1.07, 1.45], feh=0.0,   ebv=0.12, dm=9.24, xlim=(0.0, 1.5), ylim=(16, 8),   savefile='casadoiso.png'),
    dict(pattern='Theia_844',       label='Theia 844',       seismic_ages=[0.24, 0.28], lit_ages=[0.10, 0.27, 0.44], feh=0.0,   ebv=0.12, dm=9.09, xlim=(0.0, 2.0), ylim=(17, 6.5), savefile='theia844iso.png'),
]
for cfg in CLUSTERS:
    print(f"=== {cfg['label']} ===")
    color, gmag, good = get_cluster_data(df, cfg['pattern'])
    plot_isochrones(color, gmag, good, cfg['label'], cfg['seismic_ages'], cfg['lit_ages'],
                    cfg['feh'], cfg['ebv'], cfg['dm'], cfg['xlim'], cfg['ylim'], cfg['savefile'])
